In [2]:
from edgar import set_identity, Company

set_identity("prithvihegde123@gmail.com")

### Get latest 10-Q for JP Morgan Chase

In [67]:
company = Company("JPM")
latest10Q = company.get_filings(form="10-Q").head(1)

In [20]:
tenQ = latest10Q[0].obj()
print(tenQ.company, tenQ.filing_date)
print(tenQ.form)

JPMORGAN CHASE & CO 2026-08-06
10-Q


In [ ]:
# get the entire 10-Q text

document = tenQ.document

# important!! this helps find all of an objects properties and functions
[m for m in dir(document) if not m.startswith("_")] 

['chunks',
 'extract_key_information',
 'extract_section_text',
 'find_first_node',
 'find_nodes',
 'get_available_sec_sections',
 'get_sec_section',
 'get_sec_section_info',
 'get_section',
 'has_tables',
 'has_xbrl',
 'headings',
 'is_empty',
 'metadata',
 'prepare_for_llm',
 'root',
 'search',
 'sections',
 'tables',
 'text',
 'to_dataframe',
 'to_json',
 'to_markdown',
 'validate',
 'walk',
 'xbrl_facts']

In [ ]:
tenQ_text = document.text()

print(len(tenQ_text))
print(tenQ_text[:1500])

print(document.get_available_sec_sections())
# this has part i, part ii, etc. the SEC has a standard definion for these:

1236483
UNITED STATES

SECURITIES AND EXCHANGE COMMISSION

Washington, D.C. 20549

FORM 10-Q

Quarterly report pursuant to Section 13 or 15(d) of

the Securities Exchange Act of 1934

  For the quarterly period ended      Commission file
  June 30, 2026                       number               1-5805
 ───────────────────────────────────────────────────────────────────

JPMorgan Chase & Co.

(Exact name of registrant as specified in its charter)

  Delaware                                                    13-2624428
  (State or other jurisdiction of                             (I.R.S. employer


  incorporation or organization)                              identification no.)
  270 Park Avenue,
  New York,                                     New York      10017
  (Address of principal executive offices)                    (Zip Code)

Registrant’s telephone number, including area code: ( 212 ) 270-6000

Securities registered pursuant to Section 12(b) of the Act:

  Title of each clas

### Format of a 10-Q filing (general, not always true)

| Part I: Financial Information   | Part II: Other Information             |
|---------------------------------|----------------------------------------|
| Item 1: Financial Statements    | Item 1: Legal Proceedings              |
| Item 2: MD&A                    | Item 1A: Risk Factors                  |
| Item 3: Market Risk Disclosures | Item 2: Unregistered Sales of Equity   |
| Item 4: Controls and Procedures | Item 3: Defaults on Senior Securities  |
|                                 | Item 4: Mine Safety Disclosures        |
|                                 | Item 5: Other Information              |
|                                 | Item 6: Exhibits                       |


In [43]:
sections = document.get_available_sec_sections()
print(sections)

['part_i_item_1', 'part_i_item_2', 'part_i_item_3', 'part_i_item_4', 'part_ii_item_1', 'part_ii_item_1a', 'part_ii_item_2', 'part_ii_item_3', 'part_ii_item_4', 'part_ii_item_5', 'part_ii_item_6']


In [ ]:
section1 = document.get_sec_section('part_i_item_1')
print(len(section1))
print(section1[:500])
print("**************************")
print(section1[-500:])

# this gives us the Consolidated statements of income (unaudited)
# the parser for section1 might not end exactly when the statement of income ends - important to check if the content is actually what we need.

345329
JPMorgan Chase & Co.

Consolidated statements of income (unaudited)

Three months ended June 30,  Six months ended June 30,
(in millions, except per share data)  2026  2025  2026  2025
Revenue
Investment banking fees  $  3,208  $  2,499  $  6,066  $  4,677
Principal transactions  9,007  7,149  16,994  14,763
Lending- and deposit-related fees  2,511  2,248  4,905  4,380
Asset management fees  5,658  4,806  11,173  9,506
Commissions and other fees  2,614  2,194  5,096  4,227
Investment securities 
**************************
 share class” level to represent the quartile ranking for U.K., Luxembourg and Hong Kong funds and at the fund level for all other funds. The performance data may have been different if all share classes had been included. Past performance is not indicative of future results.

“Primary share class” means the C share class for European funds and Acc share class for Hong Kong and Taiwan funds. If these share classes are not available, the oldest share class is us

In [ ]:
for key in ["part_i_item_2", "part_ii_item_1a"]:
    s = document.get_sec_section(key)
    print(key, len(s))
    print(s[:500])
    print("*****")
    print(s[-500:])
    print("=" * 60)

# we see that the MD&A, and the Risk factors are extracted cleanly
# these two are the mail 'signals' or 'changes' that we will be looking for
# we see that the risk factors

part_i_item_2 274164
INTRODUCTION

The following is Management’s discussion and analysis of the financial condition and results of operations (“MD&A”) of JPMorgan Chase & Co. (“JPMorganChase” or the “Firm”) for the second quarter of 2026. 

This Quarterly Report on Form 10-Q for the second quarter of 2026 (“Form 10-Q”) should be read together with JPMorganChase’s Annual Report on Form 10-K for the year ended December 31, 2025 (“2025 Form 10-K”). Refer to the Glossary of terms and acronyms and line of business metric
*****
 and uncertainties detailed in Part I, Item 1A: Risk Factors in JPMorganChase’s 2025 Form 10-K.

Any forward-looking statements made by or on behalf of the Firm speak only as of the date they are made, and JPMorganChase does not undertake to update any forward-looking statements. The reader should, however, consult any further disclosures of a forward-looking nature the Firm may make in any subsequent Annual Reports on Form 10-K, Quarterly Reports on Form 10-Q or Curr

### Notes based on MD&A and Risk Factors:

- we see that the MD&A, and the Risk factors are extracted cleanly
- these two are the mail 'signals' or 'changes' that we will be looking for
- 10-Q MD&A happen quarterly, and this might act as a more frequent signal
- 10-Q risk factor sections are typically just cross-references to the 10-K, so risk factor changes are tracked annually

In [ ]:
# Fxn to return the MD&A text for a filing

def get_10Q_mdna(filing):
    
    if filing.form != '10-Q':
        raise ValueError(f"Expected 10-Q, got {filing.form}")

    document = filing.obj().document
    sections = document.get_available_sec_sections()

    # the MD&A is generally part_i_item_2, as per SEC guidelines
    if "part_i_item_2" not in sections:
        return None

    return document.get_sec_section("part_i_item_2")

In [ ]:
# find the period variable

filing = company.get_filings(form="10-Q").latest()
[a for a in dir(filing) if "period" in a.lower()]
# it is period_of_report


['period_of_report']

In [52]:
print(filing.period_of_report, type(filing.period_of_report))

2026-06-30 <class 'str'>


### Save the MD&A text, along with the filing period, into the Data/company folder

In [ ]:
from pathlib import Path

filing = company.get_filings(form="10-Q").latest()

mdna = get_10Q_mdna(filing)
time_period = filing.period_of_report

def save_mdna(time_period, mdna, ticker):

    if mdna is None:
        raise ValueError(f"No MD&A section found for {ticker} {time_period}")

    out_dir = Path(f"../data/processed/{ticker}")
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{time_period}_mdna.txt"
    out_path.write_text(mdna, encoding="utf-8")
    print(f"Saved {len(mdna):,} chars to {out_path}")

    return None

In [54]:
# check

lines = mdna.splitlines()
page_nums = [l for l in lines if l.strip().isdigit()]
print(len(lines), "lines,", len(page_nums), "are just numbers:", page_nums[:10])

3547 lines, 89 are just numbers: ['4', '5', '6', '7', '8', '9', '10', '11', '12', '13']


### Now, run this for JPMC last 2 years (8 quarters). 8 quarters has 6 10-Qs and 2 10-Ks.
#### Q4 doesn't have a 10-Q, it is replaced by the 10-K for the year.

In [59]:
latest_filings = company.get_filings(form="10-Q").latest(6)

In [61]:
for filing in latest_filings:
    mdna = get_10Q_mdna(filing)
    time_period = filing.period_of_report

    save_mdna(time_period, mdna, company.get_ticker())
    

Saved 274,164 chars to ../data/JPM/2026-06-30_mdna.txt
Saved 239,918 chars to ../data/JPM/2026-03-31_mdna.txt
Saved 276,621 chars to ../data/JPM/2025-09-30_mdna.txt
Saved 269,044 chars to ../data/JPM/2025-06-30_mdna.txt
Saved 234,464 chars to ../data/JPM/2025-03-31_mdna.txt
Saved 290,793 chars to ../data/JPM/2024-09-30_mdna.txt


### NOTE: we can see that there's a trend in the size of MDnA by Quarter? Check if this is just a coincidence. If not, we might have to normalize for this somehow later?

### Get the MD&A from the 10-Ks as well (covers Q4)

In [64]:
# Fxn to return the MD&A text for a 10-K filing

def get_10K_mdna(filing):
    
    if filing.form != '10-K':
        raise ValueError(f"Expected 10-K, got {filing.form}")

    document = filing.obj().document
    sections = document.get_available_sec_sections()

    # the MD&A is Item 7, which sits in Part II of a 10-K
    if "part_ii_item_7" not in sections:
        return None

    return document.get_sec_section("part_ii_item_7")

In [65]:
latest_10K_filings = company.get_filings(form="10-K").latest(2)

In [66]:
for filing in latest_10K_filings:
    mdna = get_10K_mdna(filing)
    time_period = filing.period_of_report

    save_mdna(time_period, mdna, company.get_ticker())

Saved 419,380 chars to ../data/processed/JPM/2025-12-31_mdna.txt
Saved 437,185 chars to ../data/processed/JPM/2024-12-31_mdna.txt
